<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/fabiobento/rl-course-hf/blob/main/unit2/introduction_to_qlearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/fabiobento/rl-course-hf/blob/main/unit2/introduction_to_qlearning.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

Adaptado do [Hugging Face Deep Reinforcement Learning Course](https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt)

# Introdução ao Q-Learning

## Introdução

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/thumbnail.jpg" alt="The RL process" width="100%">
<span style="font-size:80%">

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Nesta unidade, **vamos nos aprofundar em um dos métodos de RL: _value-based methods_** e estudar nosso primeiro algoritmo de RL: **Q-Learning**.

Vamos também **implementar nosso primeiro agente de RL do zero**, um agente de Q-Learning, e treiná-lo em dois ambientes:

1. Frozen-Lake-v1 (versão _non-slippery_): onde nosso agente precisará **ir do estado inicial (S) ao estado final (G)** caminhando apenas sobre blocos congelados (F) e evitando buracos (H).
2. Um táxi autônomo: onde nosso agente precisará **aprender a navegar** por uma cidade para **transportar seus passageiros do ponto A ao ponto B**.



<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/envs.gif" alt="The RL process" width="100%">
<span style="font-size:80%">

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Nessa unidade, vamos:

- Aprender sobre **value-based methods**.
- Aprender sobre as **diferenças entre Monte Carlo e Temporal Difference Learning**.
- Estudar e implementar **nosso primeiro algoritmo RL**: Q-Learning.

Esta unidade é **fundamental se você deseja trabalhar com Deep Q-Learning**: o primeiro algoritmo Deep RL que jogou jogos Atari e superou o nível humano em alguns deles (breakout, space invaders, etc).

Então, vamos começar! 🚀

## O que é RL? Uma breve revisão

Na RL, criamos um agente capaz de **tomar decisões inteligentes**.

Por exemplo, um agente que **aprende a jogar um videogame**.
Ou um agente da bolsa de valores que **aprende a maximizar seus lucros** decidindo **quais ações comprar e quando vendê-las**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit1/images/RL_process.jpg" alt="The RL process" width="100%">
<span style="font-size:80%">
O processo RL: um loop de state, action, reward e próximo state.

Fonte: <a href="http://incompleteideas.net/book/RLbook2020.pdf" target="_blank">Reinforcement Learning: An Introduction, Richard Sutton and Andrew G. Barto</a>
</span>

Para tomar decisões inteligentes, nosso agente aprenderá com o ambiente, **interagindo com ele por meio de tentativa e erro** e recebendo recompensas (positivas ou negativas) **como único feedback**.

Seu objetivo **é maximizar sua recompensa cumulativa esperada**(_expected cumulative reward_) devido à hipótese da recompensa.

**O processo de tomada de decisão do agente é chamado de _policy_ $\pi$**: dado um estado, uma _policy_ produzirá uma ação ou uma distribuição de probabilidade sobre as ações. Ou seja, dada uma observação do ambiente, uma _policy_ fornecerá uma ação (ou múltiplas probabilidades para cada ação) que o agente deve realizar.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/policy.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Nosso objetivo é encontrar uma _optimal policy_ $\pi$*, ou seja, uma _policy_ que leve à melhor recompensa cumulativa esperada.

E para encontrar essa _optimal policy_ (resolvendo assim o problema de RL), existem dois tipos principais de métodos de RL:

- _Policy-based methods_: **treinar a _policy_ diretamente** para aprender qual ação tomar em um determinado estado.
- _Value-based methods_: **treinar uma _value function_** para aprender **qual estado é mais valioso** e usar essa _value function_ **para executar a ação que leva a ele**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/two-approaches.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

E nesta unidade, **vamos nos aprofundar nos _value-based methods_**.

## Dois tipos de _value-based methods_

Nos _value-based methods_, **aprendemos uma _value function_** que **mapeia um estado para o valor esperado ao estar nesse estado**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/vbm-1.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

O valor de um estado é o **retorno descontado esperado** que o agente pode obter se **começar nesse estado e agir de acordo com nossa _policy_**.

>Mas o que significa agir de acordo com nossa _policy_? Afinal, não temos uma _policy_ em _value-based methods_, pois treinamos uma _value function_ e não uma _policy_.

Lembre-se de que o objetivo de um **agente RL é ter uma _optimal policy_ $\pi*$**.

Para encontrar a _optimal policy_, aprendemos sobre dois métodos diferentes:

- _Value-based methods_: **Indiretamente, treinando uma _value function_** que gera o valor de um estado ou de um par estado-ação. Com base nessa _value function_, nossa _policy_ tomará uma ação.

Como a _policy_ não é treinada/aprendida, **precisamos especificar seu comportamento**. Por exemplo, se quisermos uma _policy_ que, dada a _value function_, tome ações que sempre levem à maior recompensa, criaremos uma Política Gananciosa(_Greedy Policy_).

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/two-approaches-3.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">
Dado um estado, nossa action-value function (que treinamos) gera o valor de cada ação nesse estado. Então, nossa Greedy Policy pré-definida seleciona a ação que produzirá o maior valor, dado um estado ou um par de estado-ação.

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Consequentemente, qualquer que seja o método utilizado para resolver o problema, haverá uma _policy_.
No caso dos _value-based methods_, não se treina a _policy_: a _policy_ **é apenas uma função pré-definida simples** (por exemplo, a _Greedy policy_) que utiliza os valores fornecidos pela _value function_ para selecionar suas ações.

Portanto, a diferença é:
- No treinamento _policy-based_, **a _optimal policy_ (denotada por $\pi*$) é encontrada através do treinamento direto da _policy_**.
- No treinamento _value-based_, **encontrar uma _optimal value function_ (denotada por $Q*$ ou $V*$) leva a uma _optimal policy_**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/link-value-policy.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Na verdade, na maioria das vezes, em _value-based methods_, você usará uma **_Epsilon-Greedy policy_** que lida com o trade-off entre _exploration_/_expoitation_ ; falaremos sobre isso quando discutirmos o Q-Learning na segunda parte desta unidade.

Como mencionamos acima, temos dois tipos de _value-based functions_:
- _state-value function_
- _action-value function_

### _State-value function_

Escrevemos a _state-value function+ sob uma _policy_ $\pi$ desta forma:

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/state-value-function-1.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Para cada estado, a _state-value function_ gera o retorno esperado(_expected return_) se o agente **começar nesse estado** e seguir a _policy_ para sempre depois disso (para todos os futuros timesteps).

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/state-value-function-2.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">
Se considerarmos o estado com valor -7: é o retorno esperado a partir desse estado e tomando ações de acordo com nossa policy (greedy policy), então direita, direita, direita, para baixo, para baixo, direita, direita.


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

### _Action-value function_

Na _action-value function_, para cada par estado-ação, a _action-value function_ gera **o retorno esperado** se o agente começar nesse estado, realizar essa ação e seguir a _policy_ para sempre.

O valor de executar a ação $\alpha$ no estado $s$ sob a _policy_ $\pi$ é:

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/action-state-value-function-1.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/action-state-value-function-2.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Vemos que a diferença é:
- Para a _state-value function_, calcumamos **o valor do estado** $S_{t}$
- Para a _action-value function_ , calculamos **o valor do par _state-action_ $(S_{t},A_{t})$ e, portanto, o valor de executar a ação nesse estado**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/images/two-types.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">
Nota: Não preenchemos todos os pares state-action para o exemplo da action-value function.

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Em ambos os casos, independentemente da _function value_ que escolhermos (_state-value function_ ou _action-value function_), **o valor retornado é o retorno esperado (_expected return_)**.

No entanto, o problema é que, para calcular **CADA valor de um _state_ ou de um par _state-action_, precisamos somar todas as recompensas que um agente pode obter se começar nesse estado.**

Esse pode ser um processo computacionalmente caro, e **é aí que a equação de Bellman vem nos ajudar**.

## A Equação de Bellman: simplifique nossa estimativa de valor 

A equação de Bellman **simplifica o cálculo do _state value_ ou do _state-action value_**.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/bellman.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">


Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Com o que aprendemos até agora, sabemos que se calcularmos $V(S_{t})$(o valor do estado), precisamos calcular o retorno esperado se começarmos naquele estado e seguirmos a _policy_ indefinidamente.**(A policy que definimos no exemplo a seguir é _Greedy Policy+; por simplicidade, então não teremos descontos nas recompensas).**

Então para calcular $V(S_{t})$, precisamos calcular a soma das as recompensas esperadas(_expected rewards_). Portanto:

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/bellman2.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">
Para calcular o valor do Estado 1: a soma das recompensas se o agente tivesse começado nesse estado e depois seguido a greedy policy (tomando ações que levam aos melhores valores de estado) para todos os passos temporais.

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Então, para calcular o $V(S_{t+1})$, precisamos calcular o retorno se inicarmos a partir do estado $S_{t+1}$.

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/bellman3.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">
Para calcular o valor do Estado 2: a soma das recompensas se o agente tivesse começado nesse estado e, em depois, seguido a policy durante todos os passos temporais.

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>

Como você deve ter notado, estamos repetindo o cálculo do valor de diferentes estados, o que pode ser tedioso se você precisar fazer isso para cada valor de estado ou valor de ação de estado.

Em vez de calcular o retorno esperado para cada estado ou cada par estado-ação, podemos usar a **equação de Bellman**. (dica: se você sabe o que é Programação Dinâmica, isso é muito semelhante! Se você não sabe o que é, não se preocupe!)

A equação de Bellman é uma equação recursiva que funciona assim: em vez de começar cada estado desde o início e calcular o retorno, podemos considerar o valor de qualquer estado $t$ como:

**A recompensa imediata $R_{t+1}$  + o valor descontado do estado que se segue $\gamma∗V(S_{t+1})$**

<img src="https://github.com/fabiobento/rl-course-hf/raw/main/unit2/bellman4.jpg" alt="The RL process" width="100%">

<span style="font-size:80%">

Fonte: <a href="https://huggingface.co/learn/deep-rl-course/unit0/introduction?fw=pt" target="_blank">Hugging Face Deep Reinforcement Learning Course</a>
</span>